In [62]:
import functions
import importlib
importlib.reload(functions)

from functions import *

In [ ]:
#WITHOUT VARIABLE DOPPLER
C_MPS = 299_792_458.0  # speed of light [m/s]


def sample_chip_sequences_to_iq_file(
    chip_streams_01,
    distances_m,
    out_file="gps_signal_iq_fc32.dat",
    chip_rate=1.023e6,
    fs=4.0e6,
    doppler_hz=None,
    phase0=None,
    amplitudes=None,
    use_relative_delays=True,
    common_delay_s=0.0,
    chunk_size=10_000_000,
):
    """
    Multi-satellite complex baseband IQ generator.

    Writes interleaved float32 IQ samples:
        I0, Q0, I1, Q1, I2, Q2, ...

    Parameters
    ----------
    chip_streams_01 : list of arrays
        One 0/1 chip stream per satellite.
    distances_m : array-like
        One distance per satellite [m].
    out_file : str
        Output filename.
    chip_rate : float
        Chip rate [Hz].
    fs : float
        Sample rate [Hz].
    doppler_hz : array-like or None
        Per-satellite Doppler [Hz]. Defaults to zeros.
    phase0 : array-like or None
        Per-satellite initial carrier phase [rad]. Defaults to zeros.
    amplitudes : array-like or None
        Per-satellite amplitudes. Defaults to ones.
    use_relative_delays : bool
        If True, subtract the minimum geometric delay so the earliest
        satellite starts at t=0, then add common_delay_s.
    common_delay_s : float
        Extra delay added to all satellites. Useful if you want all
        pseudoranges shifted upward by a common amount.
    chunk_size : int
        Samples per write chunk.

    Returns
    -------
    n_total_samples : int
        Total number of complex samples written.
    """

    if len(chip_streams_01) == 0:
        raise ValueError("Need at least one chip stream")

    n_sats = len(chip_streams_01)

    if len(distances_m) != n_sats:
        raise ValueError("distances_m must have same length as chip_streams_01")

    # Convert chip streams from 0/1 to -1/+1
    chips = []
    for k, chips_01 in enumerate(chip_streams_01):
        arr = np.asarray(chips_01, dtype=np.uint8)
        if not np.all((arr == 0) | (arr == 1)):
            raise ValueError(f"Chip stream {k} contains values other than 0/1")
        chips.append(np.where(arr == 0, -1.0, 1.0).astype(np.float32))

    lengths = [len(c) for c in chips]
    if len(set(lengths)) != 1:
        raise ValueError(f"All chip streams must have same length, got {lengths}")

    n_chips = lengths[0]
    signal_duration_s = n_chips / chip_rate
    n_signal_samples = int(np.floor(signal_duration_s * fs))

    distances_m = np.asarray(distances_m, dtype=np.float64)
    delays_s = distances_m / C_MPS

    if use_relative_delays:
        delays_s = delays_s - np.min(delays_s)

    delays_s = delays_s + float(common_delay_s)
    delay_samples = np.round(delays_s * fs).astype(np.int64)

    if doppler_hz is None:
        doppler_hz = np.zeros(n_sats, dtype=np.float64)
    else:
        doppler_hz = np.asarray(doppler_hz, dtype=np.float64)
        if len(doppler_hz) != n_sats:
            raise ValueError("doppler_hz must have same length as chip_streams_01")

    if phase0 is None:
        phase0 = np.zeros(n_sats, dtype=np.float64)
    else:
        phase0 = np.asarray(phase0, dtype=np.float64)
        if len(phase0) != n_sats:
            raise ValueError("phase0 must have same length as chip_streams_01")

    if amplitudes is None:
        amplitudes = np.ones(n_sats, dtype=np.float32)
    else:
        amplitudes = np.asarray(amplitudes, dtype=np.float32)
        if len(amplitudes) != n_sats:
            raise ValueError("amplitudes must have same length as chip_streams_01")

    max_delay_samples = int(np.max(delay_samples))
    n_total_samples = n_signal_samples + max_delay_samples

    print("Distances [m]:", distances_m)
    print("Delays [s]:", delays_s)
    print("Delay samples:", delay_samples)
    print("Doppler [Hz]:", doppler_hz)
    print("Signal samples (undelayed):", n_signal_samples)
    print("Total complex samples:", n_total_samples)

    with open(out_file, "wb") as f:
        for start in range(0, n_total_samples, chunk_size):
            stop = min(start + chunk_size, n_total_samples)
            sample_idx = np.arange(start, stop, dtype=np.int64)

            i_sum = np.zeros(stop - start, dtype=np.float32)
            q_sum = np.zeros(stop - start, dtype=np.float32)

            for s in range(n_sats):
                src_sample_idx = sample_idx - delay_samples[s]
                valid = (src_sample_idx >= 0) & (src_sample_idx < n_signal_samples)

                if not np.any(valid):
                    continue

                # Sample the chip stream
                chip_index = np.floor(
                    src_sample_idx[valid].astype(np.float64) * chip_rate / fs
                ).astype(np.int64)
                chip_index = np.clip(chip_index, 0, n_chips - 1)
                baseband = chips[s][chip_index].astype(np.float32)

                # Complex baseband carrier with Doppler only
                t = sample_idx[valid].astype(np.float64) / fs
                phase = 2.0 * np.pi * doppler_hz[s] * t + phase0[s]

                i_sum[valid] += amplitudes[s] * baseband * np.cos(phase).astype(np.float32)
                q_sum[valid] += amplitudes[s] * baseband * np.sin(phase).astype(np.float32)

            iq = np.empty(2 * (stop - start), dtype=np.float32)
            iq[0::2] = i_sum
            iq[1::2] = q_sum
            iq.tofile(f)

    return n_total_samples

In [ ]:

#WITH VARIABLE DOPPLER

C_MPS = 299_792_458.0  # speed of light [m/s]


def sample_chip_sequences_to_iq_file_variable_doppler(
    chip_streams_01,
    distances_m,
    out_file="gps_signal_iq_fc32.dat",
    chip_rate=1.023e6,
    fs=4.0e6,
    doppler_hz_chunks=None,
    phase0=None,
    amplitudes=None,
    use_relative_delays=True,
    common_delay_s=0.0,
    chunk_size=10_000_000,
):
    """
    Multi-satellite complex baseband IQ generator with chunk-varying Doppler.

    Writes interleaved float32 IQ samples:
        I0, Q0, I1, Q1, I2, Q2, ...

    Parameters
    ----------
    chip_streams_01 : list of arrays
        One 0/1 chip stream per satellite.
    distances_m : array-like
        One distance per satellite [m].
    out_file : str
        Output filename.
    chip_rate : float
        Chip rate [Hz].
    fs : float
        Sample rate [Hz].
    doppler_hz_chunks : array-like or None
        Doppler schedule per chunk, shape:
            (n_sats, n_chunks)
        or
            (n_chunks, n_sats)
        If None, all Dopplers are zero.
    phase0 : array-like or None
        Per-satellite initial carrier phase [rad]. Defaults to zeros.
    amplitudes : array-like or None
        Per-satellite amplitudes. Defaults to ones.
    use_relative_delays : bool
        If True, subtract the minimum geometric delay so the earliest
        satellite starts at t=0, then add common_delay_s.
    common_delay_s : float
        Extra delay added to all satellites.
    chunk_size : int
        Samples per write chunk.

    Returns
    -------
    n_total_samples : int
        Total number of complex samples written.
    """

    if len(chip_streams_01) == 0:
        raise ValueError("Need at least one chip stream")

    n_sats = len(chip_streams_01)

    if len(distances_m) != n_sats:
        raise ValueError("distances_m must have same length as chip_streams_01")

    # Convert chip streams from 0/1 to -1/+1
    chips = []
    for k, chips_01 in enumerate(chip_streams_01):
        arr = np.asarray(chips_01, dtype=np.uint8)
        if not np.all((arr == 0) | (arr == 1)):
            raise ValueError(f"Chip stream {k} contains values other than 0/1")
        chips.append(np.where(arr == 0, -1.0, 1.0).astype(np.float32))

    lengths = [len(c) for c in chips]
    if len(set(lengths)) != 1:
        raise ValueError(f"All chip streams must have same length, got {lengths}")

    n_chips = lengths[0]
    signal_duration_s = n_chips / chip_rate
    n_signal_samples = int(np.floor(signal_duration_s * fs))

    distances_m = np.asarray(distances_m, dtype=np.float64)
    delays_s = distances_m / C_MPS

    if use_relative_delays:
        delays_s = delays_s - np.min(delays_s)

    delays_s = delays_s + float(common_delay_s)
    delay_samples = np.round(delays_s * fs).astype(np.int64)

    if phase0 is None:
        phase0 = np.zeros(n_sats, dtype=np.float64)
    else:
        phase0 = np.asarray(phase0, dtype=np.float64)
        if len(phase0) != n_sats:
            raise ValueError("phase0 must have same length as chip_streams_01")

    if amplitudes is None:
        amplitudes = np.ones(n_sats, dtype=np.float32)
    else:
        amplitudes = np.asarray(amplitudes, dtype=np.float32)
        if len(amplitudes) != n_sats:
            raise ValueError("amplitudes must have same length as chip_streams_01")

    max_delay_samples = int(np.max(delay_samples))
    n_total_samples = n_signal_samples + max_delay_samples

    # Number of output chunks
    n_chunks = (n_total_samples + chunk_size - 1) // chunk_size

    # Doppler schedule
    if doppler_hz_chunks is None:
        doppler_hz_chunks = np.zeros((n_sats, n_chunks), dtype=np.float64)
    else:
        doppler_hz_chunks = np.asarray(doppler_hz_chunks, dtype=np.float64)

        if doppler_hz_chunks.ndim != 2:
            raise ValueError("doppler_hz_chunks must be a 2D array")

        # Accept either (n_sats, n_chunks) or (n_chunks, n_sats)
        if doppler_hz_chunks.shape == (n_sats, n_chunks):
            pass
        elif doppler_hz_chunks.shape == (n_chunks, n_sats):
            doppler_hz_chunks = doppler_hz_chunks.T
        else:
            raise ValueError(
                f"doppler_hz_chunks must have shape ({n_sats}, {n_chunks}) "
                f"or ({n_chunks}, {n_sats}), got {doppler_hz_chunks.shape}"
            )

    print("Distances [m]:", distances_m)
    print("Delays [s]:", delays_s)
    print("Delay samples:", delay_samples)
    print("Signal samples (undelayed):", n_signal_samples)
    print("Total complex samples:", n_total_samples)
    print("Chunks:", n_chunks)
    print("Doppler schedule shape:", doppler_hz_chunks.shape)

    # Running carrier phase per satellite, so phase stays continuous across chunks
    phase_acc = phase0.astype(np.float64).copy()

    with open(out_file, "wb") as f:
        for chunk_idx, start in enumerate(range(0, n_total_samples, chunk_size)):
            stop = min(start + chunk_size, n_total_samples)
            n_chunk_samples = stop - start

            sample_idx = np.arange(start, stop, dtype=np.int64)

            i_sum = np.zeros(n_chunk_samples, dtype=np.float32)
            q_sum = np.zeros(n_chunk_samples, dtype=np.float32)

            for s in range(n_sats):
                # This chunk's Doppler for satellite s
                f_dopp = doppler_hz_chunks[s, chunk_idx]

                # Delayed source indices
                src_sample_idx = sample_idx - delay_samples[s]
                valid = (src_sample_idx >= 0) & (src_sample_idx < n_signal_samples)

                # Build chunk carrier with continuous phase over the whole chunk
                # phase[n] = phase_acc + 2*pi*f_dopp*n/fs
                if n_chunk_samples > 0:
                    n = np.arange(n_chunk_samples, dtype=np.float64)
                    phase_chunk = phase_acc[s] + 2.0 * np.pi * f_dopp * n / fs
                    cos_chunk = np.cos(phase_chunk).astype(np.float32)
                    sin_chunk = np.sin(phase_chunk).astype(np.float32)

                    # Advance stored phase to next chunk boundary
                    phase_acc[s] = (
                        phase_acc[s]
                        + 2.0 * np.pi * f_dopp * n_chunk_samples / fs
                    ) % (2.0 * np.pi)

                # Before the delayed signal begins, contribution stays zero
                if not np.any(valid):
                    continue

                # Sample the chip stream only where valid
                chip_index = np.floor(
                    src_sample_idx[valid].astype(np.float64) * chip_rate / fs
                ).astype(np.int64)
                chip_index = np.clip(chip_index, 0, n_chips - 1)
                baseband = chips[s][chip_index]

                i_sum[valid] += amplitudes[s] * baseband * cos_chunk[valid]
                q_sum[valid] += amplitudes[s] * baseband * sin_chunk[valid]

            iq = np.empty(2 * n_chunk_samples, dtype=np.float32)
            iq[0::2] = i_sum
            iq[1::2] = q_sum
            iq.tofile(f)

    return n_total_samples

In [ ]:
res3=modulo2_frames_runs(3,Z_count_start,3)[:195906250]
res6=modulo2_frames_runs(6,Z_count_start,6)[:195906250]
res9=modulo2_frames_runs(9,Z_count_start,9)[:195906250]
res16=modulo2_frames_runs(16,Z_count_start,16)[:195906250]
res21=modulo2_frames_runs(21,Z_count_start,21)[:195906250]

rho3=np.sqrt(sum((ehpm_to_ECEFlocation(3)-ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))
rho6=np.sqrt(sum((ehpm_to_ECEFlocation(6)-ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))
rho9=np.sqrt(sum((ehpm_to_ECEFlocation(9)-ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))
rho16=np.sqrt(sum((ehpm_to_ECEFlocation(16)-ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))
rho21=np.sqrt(sum((ehpm_to_ECEFlocation(21)-ECEF(buddinge[0],buddinge[1],buddinge[2]))**2))

chips_03 = np.frombuffer(res3.unpack(), dtype=np.uint8)
chips_06 = np.frombuffer(res6.unpack(), dtype=np.uint8)
chips_09 = np.frombuffer(res9.unpack(), dtype=np.uint8)
chips_16 = np.frombuffer(res16.unpack(), dtype=np.uint8)
chips_21 = np.frombuffer(res21.unpack(), dtype=np.uint8)

num_sats = 5

chip_streams_01 = [
    chips_03,
    chips_06,
    chips_09,
    chips_16,
    chips_21,
]

distances_m = np.array([
    rho3,
    rho6,
    rho9,
    rho16,
    rho21,
], dtype=float)



df = pd.read_csv("GPS_D1C_doppler.csv")

sat_data = {
    sv: {
        "time": grp["time"].tolist(),
        "doppler": grp["doppler"].tolist()
    }
    for sv, grp in df.groupby("sv")
}

doppler_list = np.array([
    sat_data["G16"]["doppler"][:375],
    sat_data["G16"]["doppler"][:375],
    sat_data["G16"]["doppler"][:375],
    sat_data["G16"]["doppler"][:375],
    sat_data["G16"]["doppler"][:375],
], dtype=float)

amplitudes = np.ones(num_sats, dtype=np.float32) * 0.2

n_total = sample_chip_sequences_to_iq_file_variable_doppler(
    chip_streams_01=chip_streams_01,
    distances_m=distances_m,
    out_file="gps_signal_iq_fc32.dat",
    chip_rate=1.023e6,
    fs=4.0e6,
    doppler_hz_chunks=doppler_list,   # shape (n_sats, 375) or (375, n_sats)
    phase0=None,
    amplitudes=amplitudes,
    use_relative_delays=True,
    common_delay_s=0.0,
    chunk_size=int(37500*20*1023/375),
)

print("complex samples:", n_samples)

Distances [m]: [20054507.79143932 21290782.26494222 23269387.89892859 25244057.78442071
 25542586.34599991]
Delays [s]: [0.         0.00412377 0.01072369 0.01731048 0.01830626]
Delay samples: [    0 16495 42895 69242 73225]
Signal samples (undelayed): 766006842
Total complex samples: 766080067
Chunks: 375
Doppler schedule shape: (5, 375)
complex samples: 766347646
